# 01 — Matched target-load latency

This view reports run-level latency percentiles at the matched 1,000 msg/s operating point. It does not reconstruct a CDF from percentile summaries or establish capacity. Canonical input requires 30 complete runs per system; explicit diagnostic input remains descriptive and non-thesis.


In [ ]:
import os, re
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.canonical import target_latency_table
from wafer_analysis.focused import evidence_label, pending_record, percentile_rows
from wafer_analysis.paths import resolve_analysis_batch
from wafer_analysis.plots import SYSTEM_COLORS, save_figure
from wafer_analysis.tables import save_table

batch, canonical = resolve_analysis_batch('e-perf-2', os.environ.get('E_PERF_2_DIR'))
raw=pd.DataFrame() if batch is None else percentile_rows(batch)
if canonical:
    records=[]
    for row in raw.to_dict('records'):
        match=re.search(r'run-(\d+)', str(row['run']))
        if match is None: raise ValueError(f"malformed canonical run name: {row['run']}")
        records.append({**row,'run_index':int(match.group(1))})
    out=target_latency_table(records)
else:
    rows=[]
    for system in ['ekuiper','native','wafer']:
        values=raw[raw.condition==system] if not raw.empty else raw
        if values.empty: rows.append(pending_record(system,'no passed percentile leaf','p95 nanoseconds'))
        else: rows.append({'condition':system,'status':'READY','N_runs':len(values),'median_p95_ns':values.p95_ns.median(),'units':'nanoseconds','estimator':'median run p95','uncertainty':'descriptive only','claim_boundary':'diagnostic matched target load; not capacity','thesis_evidence':False})
    out=pd.DataFrame(rows)
print(evidence_label(int(out.get('N_runs',pd.Series(dtype=int)).sum()), 'nanoseconds', canonical))
display(out)
ready=out[out.get('status',pd.Series(['READY']*len(out))).eq('READY')] if not out.empty else out
if not ready.empty:
    colors=[SYSTEM_COLORS[{'wafer':'WAFER','native':'Native','ekuiper':'eKuiper'}[name]] for name in ready.condition]; fig,ax=plt.subplots(); ax.bar(ready['condition'],ready['median_p95_ns']/1e6,color=colors); reference=ready.loc[ready.condition=='ekuiper','median_p95_ns']; ax.axhline(reference.iloc[0]/1e6 if not reference.empty else 0,linestyle='--',color='black',label='eKuiper reference'); ax.set_ylabel('Median run p95 (ms)'); ax.set_title('Matched 1,000 msg/s latency — run-level estimator'); ax.legend()
    output=os.environ.get('WAFER_ANALYSIS_OUTPUT_DIR')
    if output: save_figure(fig,'e-perf-2/target-load-latency',output); save_table(out,'e-perf-2-target-load-latency',output)
